Section A — Concept Application
S1: Computer Vision vs Rules-Based Filters
Question: Explain what Computer Vision is and how it differs from a simple rules-based image filter. Why is a CV-based approach more suitable for categorising diverse food images than writing fixed conditions based on colour thresholds or brightness alone?

Answer: Computer Vision (CV) uses algorithms and machine learning to extract semantic meaning, patterns, and high-level features from images. A simple rules-based filter relies on hardcoded pixel manipulation (e.g., "if red pixel intensity > 200").

A CV-based approach is superior for food categorization because food is incredibly diverse in presentation, lighting, and angles. A rules-based color threshold would easily confuse a red apple with a pepperoni pizza, whereas a CV model learns underlying textures, shapes, and structural patterns, making it highly robust against varied real-world conditions.

S2: Image Preprocessing Pipeline
Question: Describe the image preprocessing steps you would apply before passing each restaurant photo to the model. Why is resizing necessary, and in what situations would converting an RGB food image to grayscale be beneficial — and when would it lose information critical to a freshness check?

Answer: Preprocessing Steps:

Resize the image from 4000x3000 down to 128x128.
Convert the image from RGB to Grayscale.
Why resizing is necessary: The CV model's architecture expects a strict input tensor shape of 128x128. Passing a larger image causes an immediate shape mismatch error. Resizing also drastically reduces memory overhead.

Grayscale benefits: It collapses 3 color channels into 1, vastly speeding up computation. This is highly beneficial if the model only needs to look at structural shapes or edges.

Information loss: Grayscale completely destroys color data. If a freshness check relies on color cues (e.g., detecting if a green vegetable has turned brown or if meat looks grey instead of red), converting to grayscale will hide these defects and break the system.

S3: Image Transformation Chaining
Question: Which OpenCV image transformation operations would you chain together to normalise these images before passing them to an OCR engine? Justify the order in which you would apply rotation, cropping, and flipping — and explain what goes wrong if you attempt to crop before correcting the orientation.

Answer: Order of Operations:

Rotate (to correct the random angles and align the bag vertically)
Crop (to extract the lower-right quadrant containing the label)
Flip (to mirror the image if the camera angle recorded the text backwards)
Justification & Cropping Issue: You must rotate the image first so the delivery bag aligns with the true X and Y axes of the image frame. Only then will the "lower-right quadrant" actually contain the label. If you attempt to crop before rotating, you are cropping the lower-right corner of a tilted bag, which means you will likely slice the nutrition label in half or miss it entirely.

S4: Canny Edge Detection & Blurring
Question: Explain how the Canny edge detection algorithm works conceptually. Why is applying a Gaussian blur before Canny important, and what trade-off exists between the low_threshold and high_threshold parameters when tuning the detector to detect dark char marks without also highlighting every bright plate rim?

Answer: How Canny works: It calculates intensity gradients to find edges, applies non-maximum suppression to thin those edges down to 1-pixel wide lines, and finally uses hysteresis thresholding to connect strong edges and eliminate weak noise.

Why Gaussian blur is important: Edge detection calculates mathematical derivatives, making it hyper-sensitive to image noise. Blurring smooths out this noise so Canny doesn't mistake random pixel grain for actual edges.

Threshold Trade-off: Lowering the thresholds allows the detector to pick up faint, subtle edges (like dark char marks), but it will also pick up a massive amount of background noise. Raising the thresholds filters out noise (like the texture of a bright plate rim) but risks completely missing the subtle char marks. You must tune them carefully to find the sweet spot.

S5: Haar Cascade Parameters
Question: What causes false positives in Haar cascade face detection, and which parameters can you tune to reduce them? Explain the effect of increasing minNeighbors and describe one specific scenario where setting it too high would cause the module to fail in a real deployment.

Answer: False Positives: Haar cascades use simple light-and-dark contrast rectangles to detect faces. Background objects with similar contrast patterns (like a round clock, a picture frame, or wall shadows) will trick the classifier into triggering a false positive.

Tuning Parameters: scaleFactor, minNeighbors, and minSize.

Effect of minNeighbors: Increasing minNeighbors forces the algorithm to require more overlapping detection windows before confirming a face. It makes the detector much stricter, which heavily reduces false positives.

Failing Scenario: If minNeighbors is set too high, the detector becomes overly strict. In a real deployment where lighting is poor or the delivery agent is moving fast/looking slightly away, the camera will fail to detect the actual agent entirely (a false negative), locking them out.

S6: Video Processing & Frame Skipping
Question: Compare reading and processing every frame versus sampling frames at a fixed interval in OpenCV. What strategy would you use to skip redundant frames without missing important motion events, and how does the cap.set(cv2.CAP_PROP_POS_FRAMES, n) property help implement controlled frame skipping?

Answer: Comparison: Reading every frame captures absolute detail but causes massive computational lag and high storage costs. Sampling frames at intervals drastically speeds up processing and saves disk space by ignoring redundant, static data.

Strategy: Use motion detection (like absolute difference between consecutive frames or background subtraction). If the pixel delta exceeds a certain threshold, process and save the frame; otherwise, skip it.

Role of cap.set(): The cap.set(cv2.CAP_PROP_POS_FRAMES, n) property allows you to manually jump the video pointer directly to frame n. Instead of forcing OpenCV to decode and read all the useless intermediate frames into memory just to ignore them, you can perform true, computationally-efficient frame skipping.